## Run the cells below to get the data set and change the value of 'kinetics_type' to get the dataset with different kinetics

In [ ]:
import os
import json
from collections import defaultdict
import torch
from rdkit import Chem
import pandas as pd
import os
from tqdm import tqdm
kinetics_type = 'kkm'
inference = True
base_dir = f"../Data/{kinetics_type.upper()}/NewestFeature/"
all_kinetics_data_path = "../Data/all_kinetics_data.pt"
os.makedirs(base_dir, exist_ok=True)

def get_kinetics_data(kinetics_type, kinetics_data, inference=True):
    '''
    extract all data.
    '''
    def get_op(types):
        def mul(x, y):
            return float(x)*float(y)
        def div(x, y):
            return float(x)/float(y)
        if types == 'kcat':
            return mul
        else:
            return div
    def get_data(first_value_set, second_value_set, index_set, op):
        dataset = []
        for pair in index_set:
            seq, smi = pair.split('_')
            value = op(first_value_set[seq][smi][-1], second_value_set[seq][smi][-1])
            dataset.append({'seq':seq, 'smi':smi, 'value':value, 'source':[first_value_set[seq][smi][0], second_value_set[seq][smi][0]]})
        return dataset
        
    index = kinetics_data['index']
    dataset = []
    for k in index.keys():
        k_split = k.split('_')
        if kinetics_type in k_split:
            for pair in index[k]:
                seq, smi = pair.split('_')
                value = kinetics_data['kinetics_set'][kinetics_type][seq][smi][-1]
                dataset.append({'seq':seq, 'smi':smi, 'value':value, 'source':[kinetics_data['kinetics_set'][kinetics_type][seq][smi][0]]})
    if inference:
        types = ['kcat', 'km', 'kkm']
        types.remove(kinetics_type)
        first_value_set = kinetics_data['kinetics_set'][types[0]]
        second_value_set = kinetics_data['kinetics_set'][types[1]]
        index_type = '_'.join(types)+'_IndexSet'
        index_set = kinetics_data['index'][index_type]
        print(types[0], types[1], index_type)
        dataset.extend(get_data(first_value_set, second_value_set, index_set, get_op(kinetics_type)))
    return dataset
kinetics_data = torch.load(all_kinetics_data_path)
raw_data = get_kinetics_data(kinetics_type, kinetics_data, inference)
# if not os.path.exists(base_dir):
#     os.mkdir(base_dir)
print(f"total samples:{len(raw_data)}")
# exclude negative kcat value.
new_raw_data = []
for item in raw_data:
    if float(item["value"])>0 and len(item['seq']) < 1023:
        new_raw_data.append(item)
print(f"samoles:{len(new_raw_data)}, exclude {len(raw_data) - len(new_raw_data)} samples")
seq = defaultdict(lambda:len(seq))
smiles = defaultdict(lambda:len(smiles))
for item in new_raw_data:
    seq[item["seq"]]
    smiles[item["smi"]]
print("protein seqs:", len(seq))
print("mols:", len(smiles))
index_seq = {item: key for key, item in seq.items()}
index_smiles = {item: key for key, item in smiles.items()}
pair_info = []
for item in new_raw_data:
    pair_info.append([seq[item["seq"]], smiles[item["smi"]], float(item["value"]), item['source']])
# store
torch.save(pair_info, os.path.join(base_dir, 'pair_info'))
torch.save(index_seq, os.path.join(base_dir, 'index_seq'))
torch.save(index_smiles, os.path.join(base_dir, 'index_smiles'))
# seqs process
seq_str = ''
for key, item in index_seq.items():
    seq_str += f'>{key}\n{item}\n'
with open(os.path.join(base_dir, "seq_str.fasta"), 'w') as f:
    f.write(seq_str)

cmd = f'python ../Code/extract.py esm1v_t33_650M_UR90S_1 {os.path.join(base_dir, "seq_str.fasta")} {os.path.join(base_dir, "esm1v_t33_650M_UR90S_1_embeding_1280")} --repr_layers 33 --include per_tok'
os.system(cmd)

base = os.path.join(base_dir, "esm1v_t33_650M_UR90S_1_embeding_1280/")
def change(index, layer):
    data = torch.load(base+f'{index}.pt')
    data = data['representations'][layer]
    torch.save(data, base+f'{index}.pt')
file_list = os.listdir(base)
length = len(file_list)
for index in tqdm(range(length)):
    change(index, 33)

# using Unimol to generate mol-feature

In [ ]:
import torch
from unimol_tools import UniMolRepr
clf = UniMolRepr(data_type='molecule')
def trans_smiles(index_smiles_path, store_path):
    smiles = torch.load(index_smiles_path)
    n = len(smiles)
    smiles_list = []
    for i in range(n):
        smiles_list.append(smiles[i])
    reprs = clf.get_repr(smiles_list)
    torch.save(torch.tensor(reprs), store_path)
trans_smiles(f"../Data/{kinetics_type}/NewestFeature/index_smiles", f"../Data/{kinetics_type}/NewestFeature/smiles_embedding")